# Lending Club Loan Performance & Risk Analysis (2007–2018)
## Stage 2 — SQL Business Analysis

This is **Chapter 2 of 4**. `01_data_cleaning.ipynb` reduced 2.26 million raw rows and 151 variables into **1,371,165 clean records with 20 analytical features**, stored in `cleaned_data.db`. That structured dataset is the input for every query in this notebook.

SQL is the right tool for this chapter because the business questions here demand **exact, reproducible metrics at scale**: total default rates, segment-level breakdowns, group averages, and profitability by vintage cohort. SQL answers them efficiently and transparently. The visual layer — distributions, scatter plots, interaction heatmaps — comes in Chapter 3.

## Contents

This notebook uses SQL to extract precise business metrics from the cleaned loan data. Every query answers a specific risk or profitability question at portfolio scale.

1. **Dataset connection and overview** — Connect to `cleaned_data.db`, inspect the schema, preview the `loans` table, and review key fields.
2. **Default rate by category** — Default rates overall and by grade, FICO band, income level, purpose, term, and DTI band.
3. **Vintage analysis** — Default rate, credit quality, and profit by issue year (2007–2018).
4. **Summary of key SQL findings** — Cross-segment synthesis and top 3 actionable implications.

**Input:** `../data/cleaned_data.db` — produced by `01_data_cleaning.ipynb`
**Next stage:** `03_eda.ipynb` — visual confirmation and interaction-effect analysis of these findings

---

## 1. Dataset connection and overview

In this section we connect to the cleaned SQLite database (`cleaned_data.db`) and preview the `loans` table, which is the main fact table used throughout the analysis.

Some of the most important fields we will rely on are:

- **`grade`**: Lending Club credit grade assigned to the loan
- **`default`**: binary loan outcome (1 = defaulted, 0 = non-default)
- **`loan_amnt`**: original loan amount funded
- **`int_rate`**: interest rate charged on the loan
- **`annual_inc`**: borrower's annual income
- **`dti`**: debt-to-income ratio at origination
- **`purpose`**: declared purpose for the loan (e.g., debt consolidation, business)
- **`fico_score`**: borrower's FICO credit score
- **`issue_d`**: date the loan was issued
- **`credit_history_months`**: months of credit history at loan origination

In [27]:
%load_ext sql

%sql sqlite:///../data/cleaned_data.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


The query below simply pulls a small sample of rows so we can visually confirm the structure and values before computing any metrics.

In [28]:
%%sql
SELECT *
FROM loans
LIMIT 5;

 * sqlite:///../data/cleaned_data.db
Done.


grade,delinq_2yrs,inq_last_6mths,dti,annual_inc,revol_util,loan_amnt,int_rate,term,installment,total_acc,open_acc,mths_since_last_delinq,total_pymnt,default,ever_delinq,purpose_group,credit_history_months,issue_year,fico_score
C,0,1,5.909999847412109,55000.0,29.700000762939453,3600.0,13.989999771118164,36,123.02999877929688,13,7,30.0,4421.72021484375,0,1,Debt,148,2015,677.0
C,1,4,16.059999465942383,65000.0,19.200000762939453,24700.0,11.989999771118164,36,820.280029296875,38,22,6.0,25679.66015625,0,1,Business,192,2015,717.0
B,0,0,10.779999732971191,63000.0,56.20000076293945,20000.0,10.779999732971191,60,432.6600036621094,18,6,31.0,22705.919921875,0,0,Home,184,2015,697.0
F,1,3,25.3700008392334,104433.0,64.5,10400.0,22.450000762939453,60,289.9100036621094,35,12,12.0,11740.5,0,1,Consumer,210,2015,697.0
C,0,0,10.199999809265137,34000.0,68.4000015258789,11950.0,13.4399995803833,36,405.17999267578125,6,5,31.0,13708.9501953125,0,0,Debt,338,2015,692.0


The query below inspect column names and types.

In [29]:
%%sql
PRAGMA table_info(loans);

 * sqlite:///../data/cleaned_data.db
Done.


cid,name,type,notnull,dflt_value,pk
0,grade,TEXT,0,None,0
1,delinq_2yrs,INTEGER,0,None,0
2,inq_last_6mths,INTEGER,0,None,0
3,dti,FLOAT,0,None,0
4,annual_inc,FLOAT,0,None,0
5,revol_util,FLOAT,0,None,0
6,loan_amnt,FLOAT,0,None,0
7,int_rate,FLOAT,0,None,0
8,term,SMALLINT,0,None,0
9,installment,FLOAT,0,None,0


---

## 2. Default rates

We quantify **how often loans default** across the portfolio and key borrower and loan segments to support risk-based pricing, exposure limits, and profitability analysis.

**Metrics in this section:**

- **Portfolio default rate** — Overall default rate and volume.
- **By credit grade** — Default rate from A (lowest risk) to G (highest risk).
- **By FICO band** — Risk by credit-score tier (Poor / Fair / Good / Excellent).
- **By income level** — Default rate by borrower income band.
- **By purpose** — Risk by stated use (e.g. Debt, Home, Business, Consumer).
- **By term** — Risk by loan term (36 or 60 months).
- **By DTI band** — Risk by affordability tier (Low / Medium / High / Very high DTI).

### 2.1 Total default rate

In [30]:
%%sql
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate_pct
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_loans,total_defaults,default_rate_pct
1371165,294415,21.47


**Takeaway:** The portfolio default rate is **21.5%** — approximately 1 in 5 loans ends in default. With 294k+ defaults across 1.37M loans, this is not a tail risk; it is a core driver of portfolio returns and loss provisioning.

**Why it matters:** At 21.5%, the portfolio sits well above typical prime consumer credit default rates. Effective risk segmentation — by grade, FICO, DTI, and term — is essential to separate the high-performing segments from the loss-making ones. The breakdowns below show exactly where risk is concentrated.

### 2.2 Default Rate by Credit Grade

In [31]:
%%sql
SELECT 
    grade,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*), 2) AS default_rate_pct,
    ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,loans,defaults,default_rate_pct,pct_of_portfolio
A,236758,15869,6.7,17.3
B,398528,58357,14.64,29.1
C,390724,94687,24.23,28.5
D,206696,66797,32.32,15.1
E,96224,38609,40.12,7.0
F,32828,15261,46.49,2.4
G,9407,4835,51.4,0.7


**Takeaway:** Credit grade is the strongest single risk differentiator in this portfolio — default rate rises monotonically from **6.7% (Grade A)** to **51.4% (Grade G)**, a spread of nearly 45 percentage points.

**Why it matters:** Grades B and C together hold 57% of the portfolio by volume and see default rates of 14.6% and 24.2% respectively — making them the primary drivers of total defaults by sheer scale. Subprime grades D–G contribute disproportionately to losses despite representing only 23% of loans.

### 2.3 Default Rate by FICO Score

In [32]:
%%sql
SELECT 
    CASE
        WHEN fico_score < 600 THEN 'Poor'
        WHEN fico_score < 700 THEN 'Fair'
        WHEN fico_score < 750 THEN 'Good'
        ELSE 'Excellent'
    END AS credit_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY credit_group
ORDER BY CASE credit_group
    WHEN 'Poor' THEN 1 WHEN 'Fair' THEN 2 WHEN 'Good' THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate_pct,pct_of_portfolio
Fair,837712,210540,25.13,61.1
Good,425361,72911,17.14,31.0
Excellent,108092,10964,10.14,7.9


**Takeaway:** Default rate falls sharply as FICO improves — from **25.1% (Fair, 600–699)** down to **10.1% (Excellent, 750+)**. Fair-FICO borrowers represent 61% of portfolio volume and account for the majority of defaults, making them the highest-leverage segment for tightening underwriting. The "Good" band (700–749) sits in a meaningful middle: 17.1% default rate with a large share of volume — a key segment for risk-based pricing adjustments.

**Why it matters:** The Fair band alone holds 837K loans with a 25.1% default rate; a 1 pp reduction in that rate through stricter eligibility or better pricing would prevent roughly 8,400 defaults. The 15 pp gap between Fair and Excellent is not a statistical artifact — it reflects a structurally different borrower profile, and treating these two groups under the same underwriting criteria systematically misprices risk.

> **Note:** No loans with FICO below 600 (Poor) appear in the dataset. This reflects Lending Club's minimum credit score threshold at origination — a form of selection bias to keep in mind when benchmarking against the broader credit market.

### 2.4 Default Rate by Income Level

In [33]:
%%sql
SELECT 
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY income_group
ORDER BY CASE income_group
    WHEN 'Low Income' THEN 1 WHEN 'Middle Income' THEN 2 ELSE 3 END;

 * sqlite:///../data/cleaned_data.db
Done.


income_group,loans,defaults,default_rate_pct,pct_of_portfolio
Low Income,214632,54213,25.26,15.7
Middle Income,675851,151339,22.39,49.3
High Income,480682,88863,18.49,35.1


**Takeaway:** Default rate is inversely correlated with income — **25.3% (Low, <$40K)** vs. **18.5% (High, $80K+)** — but the spread is narrower than for grade or FICO, suggesting income alone is a weak standalone predictor.

**Why it matters:** The Middle Income band ($40K–$80K) holds 49% of all loans and a 22.4% default rate — nearly at the portfolio average. Income is most useful as a **combinatorial signal**: a low-income, high-DTI borrower in Grade D is a fundamentally different risk than a low-income borrower in Grade A. Income in isolation understates the risk concentration at the bottom.

Loan **purpose** drives both demand and risk: debt consolidation, home improvement, and business use have different default profiles and portfolio implications. Segmenting by purpose helps set product-level limits and pricing.

**Metrics in this section:**

- **Default rate by purpose** — Risk by stated use (e.g. Debt, Home, Business, Consumer).
- **Profit by purpose** — Profit or loss by purpose to see which uses are profitable.

### 2.5 Default Rate by Purpose Group

In [35]:
%%sql
SELECT 
    purpose_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY purpose_group
ORDER BY default_rate_pct DESC;

 * sqlite:///../data/cleaned_data.db
Done.


purpose_group,loans,defaults,default_rate_pct,pct_of_portfolio
Business,16777,5229,31.17,1.2
Health/Education,16225,3788,23.35,1.2
Other,79791,18301,22.94,5.8
Debt,1095457,234854,21.44,79.9
Consumer,66159,13155,19.88,4.8
Home,96756,19088,19.73,7.1


**Takeaway:** Default rates vary meaningfully by purpose. **Business loans** carry the highest default rate at **31.2%** — nearly 10 percentage points above the portfolio average. **Debt consolidation (Debt)** dominates volume at 79.9% of all loans, with a 21.4% default rate near the portfolio mean.

**Why it matters:** Business and Health/Education loans are small in volume but high in risk; Consumer and Other are mid-tier. The outsized weight of Debt consolidation means that even small changes in its default rate have large portfolio-level consequences.

**Term** (36 vs 60 months) and **debt-to-income (DTI)** are key levers: longer terms increase exposure and often risk; higher DTI indicates less capacity to absorb shocks. Segmenting by term and DTI supports underwriting and pricing.

**Metrics in this section:**

- **Default rate by term** — Risk for 36- vs 60-month loans.
- **Default rate by DTI band** — Risk by affordability tier (Low / Medium / High / Very high DTI).

### 2.6 Default Rate by Term

In [37]:
%%sql
SELECT 
    term,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY term
ORDER BY term;

 * sqlite:///../data/cleaned_data.db
Done.


term,loans,defaults,default_rate_pct,pct_of_portfolio
36,1035787,178296,17.21,75.5
60,335378,116119,34.62,24.5


**Takeaway:** 60-month loans default at **34.6%** — exactly **2× the rate of 36-month loans (17.2%)**. Despite representing only 24.5% of portfolio volume, they account for a disproportionate share of defaults due to both higher default probability and longer exposure windows.

**Why it matters:** Longer terms give borrowers more time to experience income shocks, job loss, or financial distress. The 2× default rate differential is large enough to justify distinct pricing and underwriting policies by term — treating 36 and 60-month loans as the same product understates risk.

**Recommendation:** Apply a term premium in interest rate pricing for 60-month loans. In dashboards, always segment default rate by term as a baseline filter — a portfolio that shifts toward longer terms will see its aggregate default rate rise even if borrower quality is unchanged (a mix effect).

### 2.7 Default Rate by DTI Band

In [38]:
%%sql
SELECT 
    CASE 
        WHEN dti < 10 THEN "Low"
        WHEN dti < 20 THEN "Medium"
        WHEN dti < 30 THEN "High"
        ELSE "Very high"
    END AS dti_band,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY dti_band
ORDER BY CASE dti_band 
    WHEN "Low" THEN 1 WHEN "Medium" THEN 2 WHEN "High" THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


dti_band,loans,defaults,default_rate_pct,pct_of_portfolio
Low,250260,41120,16.43,18.3
Medium,571897,109914,19.22,41.7
High,416700,102223,24.53,30.4
Very high,132308,41158,31.11,9.6


**Takeaway:** DTI shows a clear monotonic relationship with default risk — rising from **16.4% (Low DTI, <10)** to **31.1% (Very High DTI, 30+)**. The step from High to Very High is the largest single jump (+6.6 pp), signaling a nonlinear risk cliff at the top of the DTI range.

**Why it matters:** DTI measures how much of a borrower's income is already committed to debt repayments. Very High DTI borrowers have little financial buffer — a modest income disruption can trigger default. This segment defaults nearly twice as often as Low DTI borrowers, making DTI a strong affordability signal that complements FICO and grade.

---

## 3. Vintage analysis

A **vintage** is a cohort of loans originated in the same year. Comparing default rates across vintages answers a question that no single-point analysis can: **did the platform's credit quality improve or deteriorate over time?** Rising default rates in later vintages — even at similar grades — indicate looser underwriting standards or adverse borrower selection. Falling rates point to tightening eligibility criteria or more favourable economic conditions.

This section tracks Lending Club's full 2007–2018 origination history: from the conservative post-crisis years through the rapid growth phase of 2013–2015, and into the high-default cohorts of 2016–2018.

**Metrics in this section:**

- **Default rate and volume by issue year** — How each annual cohort performed relative to the 21.5% portfolio average, and how origination volume scaled across the period.
- **Credit quality and loan size by issue year** — Average FICO score, interest rate, and loan amount per vintage to identify the underwriting standard shifts that explain the default trend.
- **Profit by issue year** — Total and per-loan profit by vintage to quantify when the platform's unit economics turned negative and whether the growth phase was financially sustainable.

### 3.1 Default rate by issue year

In [39]:
%%sql
SELECT
    issue_year,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY issue_year
ORDER BY issue_year;

 * sqlite:///../data/cleaned_data.db
Done.


issue_year,loans,defaults,default_rate_pct,pct_of_portfolio
2007,251,45,17.93,0.0
2008,1562,247,15.81,0.1
2009,4716,594,12.6,0.3
2010,11536,1487,12.89,0.8
2011,21721,3297,15.18,1.6
2012,53367,8644,16.2,3.9
2013,134807,21027,15.6,9.8
2014,223510,41569,18.6,16.3
2015,377184,77442,20.53,27.5
2016,298552,73700,24.69,21.8


**Takeaway:** Post-crisis vintages (2009–2010) are the cleanest in the portfolio, defaulting at just **12.6–12.9%** — roughly half the rate of recent cohorts. From 2013 onward, default rates climbed steadily, reaching **27.2% by 2017** and holding there in 2018. Volume peaked in 2015 with 377K loans (27.5% of the entire portfolio), already originating at 20.5%.

**Why it matters:** The rising default trajectory from 2014 onward tracks Lending Club's rapid growth phase — origination volume grew 2.8× in two years (134K loans in 2013 → 377K in 2015). Rapid platform growth often requires loosening credit criteria to fill volume, and the default data is consistent with that pattern. **Critical caveat for reporting:** The 2017–2018 vintages (27.2% each) have had less time to fully season — their eventual default rates will likely be *higher* than what the data currently shows.

### 3.2 Credit quality and loan size by issue year

In [44]:
%%sql
SELECT
    issue_year,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(AVG(fico_score), 0) AS avg_fico,
    ROUND(AVG(loan_amnt), 0) AS avg_loan_amnt,
    ROUND(AVG(int_rate), 2) AS avg_int_rate
FROM loans
GROUP BY issue_year
ORDER BY issue_year;

 * sqlite:///../data/cleaned_data.db
Done.


issue_year,loans,defaults,avg_fico,avg_loan_amnt,avg_int_rate
2007,251,45,717.0,8842.0,10.32
2008,1562,247,712.0,9213.0,11.16
2009,4716,594,719.0,9847.0,12.19
2010,11536,1487,717.0,10586.0,11.75
2011,21721,3297,717.0,12048.0,12.22
2012,53367,8644,703.0,13462.0,13.64
2013,134807,21027,697.0,14707.0,14.53
2014,223510,41569,694.0,14591.0,13.66
2015,377184,77442,695.0,14664.0,12.4
2016,298552,73700,697.0,14496.0,13.12


**Takeaway:** Three trends emerge when credit quality metrics are layered onto vintage volume:

1. **Average FICO declined sharply during the growth surge** — from **717–719 (2007–2010)** down to **694 (2014)**, a 23-point drop that coincides exactly with the volume explosion from 11K to 223K loans per year. This is the clearest signal of systematic underwriting loosening in the data.

2. **Average loan amount nearly doubled** over the period — from **$8,842 (2007)** to a plateau of ~**$14,500–$15,200 (2013–2018)**. Higher average ticket sizes amplify loss-given-default even when default rates hold steady.

3. **Interest rates did not compensate for the FICO decline** — rates peaked at **14.53% in 2013** then fell to **12.40% by 2015**, the exact year volume peaked. Lower rates on weaker borrowers at peak origination volume is a combination that produced high-default vintages in 2016–2018.

**Why it matters:** The FICO trend is the key diagnostic finding of this vintage analysis. It shows that the 2016–2018 default surge was not purely driven by macroeconomic conditions — it was partly self-inflicted through credit standard loosening during the growth phase. The partial FICO recovery in 2017–2018 (701–709) suggests the platform tightened after the damage was done, but the lagged effect of 2014–2015 originations continues to show up in default rates.

### 3.3 Profit by issue year

In [41]:
%%sql
SELECT
    issue_year,
    COUNT(*) AS loans,
    ROUND(SUM(total_pymnt - loan_amnt), 0) AS profit,
    ROUND(AVG(total_pymnt - loan_amnt), 2) AS avg_profit_per_loan,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY issue_year
ORDER BY issue_year;

 * sqlite:///../data/cleaned_data.db
Done.


issue_year,loans,profit,avg_profit_per_loan,default_rate_pct
2007,251,-6569.0,-26.17,17.93
2008,1562,-324414.0,-207.69,15.81
2009,4716,3912424.0,829.61,12.6
2010,11536,8517271.0,738.32,12.89
2011,21721,25644014.0,1180.61,15.18
2012,53367,81703338.0,1530.97,16.2
2013,134807,259584946.0,1925.6,15.6
2014,223510,287207414.0,1284.99,18.6
2015,377184,190563694.0,505.23,20.53
2016,298552,-92834012.0,-310.95,24.69


**Takeaway:** Profit per vintage follows a clear arc — rising through the growth phase, peaking in 2013–2014, then collapsing as underwriting quality deteriorated. **2013 was the golden vintage**: $259M total profit at $1,926 per loan. **2014 was the volume peak in profitability**: $287M total, but avg profit had already fallen to $1,285/loan — a signal that quality was thinning. By 2015, despite originating 377K loans (the most in any single year), avg profit per loan had collapsed to just **$505** — a 74% drop from 2013. From 2016 onward the platform turned loss-making: **−$93M (2016)**, **−$292M (2017)**, **−$214M (2018)**, with avg loss per loan reaching **−$3,283 in 2018**.

**Why it matters:** This is the clearest quantification of when the growth-at-the-expense-of-quality strategy began destroying value. The profit-per-loan metric strips out volume effects and shows the underlying unit economics: each loan originated in 2017 cost the platform $1,635 on average. When combined with the FICO decline observed in the credit quality analysis, the causal chain is complete — loosened underwriting → weaker borrowers → higher defaults → losses that interest income cannot cover.

> **Maturity bias — read carefully:** The 2017–2018 losses are overstated. Loans in these vintages, especially 60-month terms, have not yet collected all scheduled payments by the dataset's cutoff. A 2018 loan may have made only 12–18 of its 60 payments, so `total_pymnt` is lower than the eventual figure. **The direction is correct — these are loss-making cohorts — but the magnitude is inflated.**

---

## 4. Summary of key SQL findings

| Segment | Key metric | Business translation |
|---|---|---|
| Portfolio overall | **21.47% default rate** | ~1 in 5 loans defaults across 1.37M loans |
| Grade A vs Grade G | **6.7% → 51.4%** | Grade is the strongest single risk differentiator — G loans are 3× more likely to default than A loans |
| FICO Fair vs Excellent | **25.1% → 10.1%** | Fair FICO (61% of volume) drives the majority of portfolio defaults |
| Income Low vs High | **25.3% → 18.5%** | Income spread is modest; most useful when combined with DTI and grade |
| Purpose: Debt | **$302M profit** | Backbone of portfolio profitability — 79.9% of volume, near-average default rate |
| Purpose: Business | **31.2% default, −$11.5M** | Highest-risk purpose group; current pricing does not cover losses |
| Term 36 vs 60 months | **17.2% → 34.6%** | 60-month loans default at exactly 2× the rate of 36-month loans |
| DTI Low vs Very High | **16.4% → 31.1%** | Clear monotonic risk gradient; steep cliff above DTI 30 |
| Vintage 2013 (peak) | **$1,926 avg profit/loan** | Best unit economics in the portfolio — low default rate at high volume; the benchmark vintage |
| Vintage 2015 | **$505 avg profit/loan** | Per-loan profit fell 74% from the 2013 peak as credit standards loosened and FICO declined |
| Vintage 2016–2017 | **−$311 → −$1,635/loan** | Unit economics turned negative; platform originated at a loss for two consecutive years |

### 4.1 Top 3 actionable implications

1. **Tighten underwriting at the Grade D–G / Fair FICO / High DTI intersection:** This segment is small in volume but responsible for a disproportionate share of defaults. Stricter eligibility criteria or materially higher rates are required to price the risk correctly.

2. **Investigate the 2016 credit inflection before it fully matures:** FICO scores declined, interest rates rose, and per-loan profit dropped from $1,926 (2013) to negative by 2016 — a three-year deterioration that preceded the loss turn. The 2016–2017 vintages still have unmatured loans; final losses may be higher than currently recorded.

3. **Review Business and Consumer loan pricing — and protect Debt consolidation:** Both Business and Consumer purpose groups are loss-making at current rates. Meanwhile, Debt consolidation generates the entirety of net portfolio profit. Any grade-mix shift within that segment is the leading indicator of portfolio deterioration and should be monitored as a primary KPI.

---

> **Bias awareness:** Results reflect only approved and funded loans (selection bias). No loans with FICO < 600 exist in the dataset (minimum underwriting threshold). Recent vintages (2017–2018) have shorter performance windows and likely understate mature default rates (temporal bias). Interpret all rates in this context.

---

**What SQL cannot show — `03_eda.ipynb` picks up here**

These queries quantify risk one dimension at a time. Three questions remain unanswered:

- **Do risk factors compound?** A Grade G borrower with Very High DTI and Fair FICO defaults at ~75% — 3.5× the portfolio average — but no single query here surfaces that interaction.
- **Are predictors redundant?** Grade and interest rate are near-perfectly correlated. Using both in a model or scorecard adds collinearity without adding signal.
- **What do the raw distributions look like?** Whether loan amounts are skewed, whether FICO has a hard floor, how DTI concentrates — these questions need histograms, not aggregates.

Notebook 3 addresses all three through visualization and bivariate interaction analysis.